# Ground Water Modeling Project Summary

This notebook summarizes the current state of the project:

- sample-point extraction for the Williston Basin test location
- vertical profile plotting
- continental batch workflow at 0.25 by 0.25 degree spacing
- provisional pore-volume integration in the top 10 km
- compact SQLite master storage and per-location extraction

The workflow is currently a first-pass prototype. The geological geometry comes from the USGS National Crustal Model geologic framework in the workspace, while porosity, conductivity, and seismic properties are provisional lithology-based defaults intended for iteration rather than final interpretation.

## Workspace Files

Core scripts created so far:

- `scripts/extract_usgs_point_profile.py`
- `scripts/plot_point_profile.py`
- `scripts/run_conus_porosity_workflow.py`
- `scripts/extract_location_from_master.py`

Representative outputs:

- `outputs/point_profile_lat_47p5000N_lon_102p0000W.json`
- `outputs/point_profile_lat_47p5000N_lon_102p0000W_layers.csv`
- `outputs/point_profile_lat_47p5000N_lon_102p0000W_profile.png`
- `outputs/conus_0p25_sqlite_test/conus_0p25_master.sqlite`
- `outputs/conus_0p25_sqlite_test/conus_0p25_aggregate.json`
- `outputs/conus_0p25_sqlite_test/extract_lat_47p5000_lon_-102p0000.json`

In [ ]:
from pathlib import Path

root = Path('.')
paths = [
    root / 'scripts' / 'extract_usgs_point_profile.py',
    root / 'scripts' / 'plot_point_profile.py',
    root / 'scripts' / 'run_conus_porosity_workflow.py',
    root / 'scripts' / 'extract_location_from_master.py',
    root / 'outputs' / 'point_profile_lat_47p5000N_lon_102p0000W.json',
    root / 'outputs' / 'point_profile_lat_47p5000N_lon_102p0000W_profile.png',
    root / 'outputs' / 'conus_0p25_sqlite_test' / 'conus_0p25_master.sqlite',
]

for path in paths:
    print(f'{path}:', 'exists' if path.exists() else 'missing')

## 1. Sample Point: Williston Basin

The first prototype step used a single test location in the Williston Basin:

- latitude: `47.5 N`
- longitude: `102.0 W`

The point-extraction script samples the nearest NCM geologic profile, decodes lithology and age names, and writes:

- JSON summary
- layer table CSV

The default output filename now includes the input coordinates so repeated runs do not overwrite one another.

In [ ]:
import json

point_json = root / 'outputs' / 'point_profile_lat_47p5000N_lon_102p0000W.json'
profile = json.loads(point_json.read_text())

print('Requested point:', profile['requested_point'])
print('Nearest grid point:', profile['nearest_grid_point'])
print('Elevation summary:')
for key, value in profile['elevation_summary_m'].items():
    print(f'  {key}: {value}')
print('Layer count:', len(profile['layers']))

## 2. Profile Plotting

A plotting script was added to visualize the extracted vertical profile.

The current figure has two panels:

- left: full profile, only thick layers labeled
- right: top 10 km zoom, all visible layers marked using an external callout column

The y-axis is elevation relative to sea level.

In [ ]:
from IPython.display import Image, display

plot_path = root / 'outputs' / 'point_profile_lat_47p5000N_lon_102p0000W_profile.png'
display(Image(filename=str(plot_path)))

## 3. Continental Batch Workflow

The next prototype step generalizes from one point to many points across CONUS.

Design choices implemented:

- grid spacing: `0.25 x 0.25` degree
- domain: CONUS bounds from the USGS geologic framework metadata
- profile lookup: fast interpolation of `Index j grid` and `Index k grid` from `NCM_SpatialGrid.nc`, followed by a small local nearest-neighbor refinement
- integration target: pore volume and fluid volume in the top 10 km below local surface
- future benchmark output: one 1D model per location stored in a compact master database

The geology comes directly from the USGS framework. The rock properties are still provisional.

## 4. Provisional Property Model

The current batch workflow assigns lithology-based placeholder properties to each interval:

- porosity fraction
- fluid saturation fraction
- conductivity in S/m
- `Vp` in km/s
- `Vs` in km/s

This is useful for testing the pipeline, but it is not a calibrated geophysical or hydrogeologic model. The next scientific upgrade should replace the internal defaults with a curated property table tied to lithology, burial depth, temperature, salinity, and possibly age or basin context.

In [ ]:
import sqlite3
import pandas as pd

db_path = root / 'outputs' / 'conus_0p25_sqlite_test' / 'conus_0p25_master.sqlite'
conn = sqlite3.connect(db_path)
properties = pd.read_sql_query('SELECT * FROM rock_properties ORDER BY rock_type_name', conn)
properties.head(20)

## 5. Compact Master Store

The original idea of writing one CSV model per location proved slow on the Google Drive-backed workspace. The workflow was therefore refactored to write a single SQLite master database.

Current tables:

- `location_summaries`
- `location_intervals`
- `rock_properties`
- `run_metadata`

This makes the continental workflow more practical and gives a stable container for later seismic and MT forward modeling.

In [ ]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
    conn,
)
tables

In [ ]:
summary_df = pd.read_sql_query(
    'SELECT * FROM location_summaries ORDER BY latitude, longitude',
    conn,
)
summary_df

## 6. Individual Extraction Tool

A separate utility was added to pull an individual location back out of the SQLite master store and save it to JSON plus interval CSV.

This is useful for:

- checking one location by hand
- passing one 1D model to a later forward solver
- comparing specific sites without scanning the full database

In [ ]:
extract_json = root / 'outputs' / 'conus_0p25_sqlite_test' / 'extract_lat_47p5000_lon_-102p0000.json'
extract_payload = json.loads(extract_json.read_text())

print('Summary:')
for key, value in extract_payload['summary'].items():
    print(f'  {key}: {value}')

print('\nFirst 5 intervals:')
for interval in extract_payload['intervals'][:5]:
    print(interval)

## 7. Aggregate Result from the Verified Test Run

A small six-point subset around the Williston Basin point was used to verify the compact workflow. The aggregate JSON is not a continental result yet; it is a correctness check for the pipeline.

In [ ]:
aggregate_path = root / 'outputs' / 'conus_0p25_sqlite_test' / 'conus_0p25_aggregate.json'
aggregate = json.loads(aggregate_path.read_text())
aggregate

## 8. Commands Used

Sample point extraction:

```bash
/opt/anaconda3/bin/python scripts/extract_usgs_point_profile.py --lat 47.5 --lon -102.0
```

Sample point plotting:

```bash
/opt/anaconda3/bin/python scripts/plot_point_profile.py --lat 47.5 --lon -102.0
```

Compact batch workflow, example subset:

```bash
/opt/anaconda3/bin/python scripts/run_conus_porosity_workflow.py --output-dir outputs/conus_0p25_sqlite_test --lat-min 47.25 --lat-max 47.75 --lon-min -102.25 --lon-max -102.0 --step-deg 0.25
```

Extract one location from the master store:

```bash
/opt/anaconda3/bin/python scripts/extract_location_from_master.py --db outputs/conus_0p25_sqlite_test/conus_0p25_master.sqlite --lat 47.5 --lon -102.0
```

## 9. Current Status

Completed:

- point-based geologic profile extraction
- point-based profile plotting
- scalable 0.25 degree CONUS batch design
- provisional top-10-km pore-volume integration
- compact SQLite master storage
- per-location extraction from the compact store

Not yet complete:

- full CONUS production run to completion
- calibrated porosity / conductivity / seismic property table
- downstream seismic and MT forward calculations
- benchmarking against independent hydrogeologic or geophysical constraints

## 10. Recommended Next Steps

1. Replace the provisional internal property defaults with a user-editable rock-property table.
2. Run the full CONUS job against the compact SQLite output path.
3. Add a small forward-model interface that reads one location from the SQLite store and exports the exact 1D seismic / conductivity model format required by future solvers.
4. Add map-based QC plots of integrated pore volume, sediment thickness, and nearest-grid-point offset.

In [ ]:
conn.close()